# Vector RAG vs Graph RAG on Noisy Company Documents

This notebook uses a small set of realistic company-style documents: onboarding notes, procurement policy, an incident retrospective, a launch brief, and a support playbook.

The goal is not to prove that vector retrieval is always bad. The goal is narrower and more honest:

- show why a **naive document-level vector baseline** performs poorly on long, mixed enterprise documents
- show why **graph-oriented retrieval** becomes the next step when answers depend on dependencies, ownership, escalation paths, and fallback routes

Both retrieval methods use the same source corpus. Ollama is used locally for embeddings and answer generation.

## 1. Why this comparison is intentionally harsh on vector retrieval

Enterprise knowledge bases usually contain documents like:

- onboarding handbooks with product notes buried inside HR guidance
- policy documents with supplier dependencies mixed with compliance language
- retrospectives that mix business impact, operations, and finance commentary

A common first implementation is to embed each long document as a single vector and retrieve the top one or two matches. That baseline is simple, but it often breaks on questions that require **multi-hop reasoning across several documents**.

This notebook uses exactly that weak baseline for vector RAG. That is deliberate. It mirrors a real early-stage implementation mistake.

Graph RAG gets an advantage here because we extract relationships from prose and query those relationships directly.

## 2. Architecture sketches

Naive vector baseline:

```mermaid
flowchart LR
    A[User Query] --> B[Ollama Embedding]
    B --> C[Similarity Search Over Full Documents]
    D[Long Company Docs] --> E[One Embedding Per Document]
    E --> C
    C --> F[Top 1 Document]
    F --> G[Ollama Generation]
    G --> H[Answer]
```

Graph-oriented retrieval:

```mermaid
flowchart LR
    A[User Query] --> B[Seed Entity Matching]
    B --> C[Graph Traversal]
    D[Relationships Extracted From Prose] --> C
    C --> E[Connected Facts Across Documents]
    E --> F[Ollama Generation]
    F --> G[Answer]
```

The key difference is not embeddings versus no embeddings. The key difference is **semantic similarity over whole documents** versus **explicit dependency traversal over extracted relationships**.

## 3. Local setup

Run these commands before executing the notebook:

```bash
pip install -r requirements.txt
ollama pull llama3.1:8b
ollama pull nomic-embed-text
ollama serve
```

This notebook assumes Ollama is available at `http://localhost:11434`.

In [ ]:
from pathlib import Path
import re
import textwrap
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_colwidth", 140)
plt.style.use("ggplot")

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OLLAMA_BASE_URL = "http://localhost:11434"
GENERATION_MODEL = "gpt-oss:20b"
EMBEDDING_MODEL = "nomic-embed-text"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Expected a data directory at {DATA_DIR}. Open the notebook from the vector_vs_graph_rag folder."
    )

print(f"Notebook directory: {BASE_DIR}")
print(f"Corpus directory:   {DATA_DIR}")

In [ ]:
def ollama_healthcheck(base_url: str = OLLAMA_BASE_URL) -> dict:
    """Confirm that the local Ollama server is reachable and models are present."""
    response = requests.get(f"{base_url}/api/tags", timeout=30)
    response.raise_for_status()
    payload = response.json()
    model_names = [item["name"] for item in payload.get("models", [])]
    return {
        "available_models": model_names,
        "embedding_model_present": EMBEDDING_MODEL in model_names,
        "generation_model_present": GENERATION_MODEL in model_names,
    }


def ollama_embed(text: str, model: str = EMBEDDING_MODEL, base_url: str = OLLAMA_BASE_URL) -> np.ndarray:
    """Create one embedding vector using the local Ollama embeddings endpoint."""
    response = requests.post(
        f"{base_url}/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=120,
    )
    response.raise_for_status()
    return np.array(response.json()["embedding"], dtype=float)


def ollama_generate(prompt: str, model: str = GENERATION_MODEL, base_url: str = OLLAMA_BASE_URL) -> str:
    """Generate a grounded answer through the local Ollama generation endpoint."""
    response = requests.post(
        f"{base_url}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=180,
    )
    response.raise_for_status()
    return response.json()["response"].strip()


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)


health = ollama_healthcheck()
health

## 4. Load the enterprise-style corpus

These are plain text documents, but they are intentionally noisy. The facts we care about are buried inside broader operational writing.

In [ ]:
def parse_document(path: Path) -> dict:
    """Parse each text file into a title and body."""
    raw_text = path.read_text(encoding="utf-8")
    lines = [line.rstrip() for line in raw_text.splitlines()]
    title = path.stem
    body_lines = []

    for line in lines:
        if line.startswith("Title:"):
            title = line.replace("Title:", "", 1).strip()
            continue
        if line.strip():
            body_lines.append(line.strip())

    body = "\n".join(body_lines)
    return {
        "doc_id": path.name,
        "title": title,
        "body": body,
        "full_text": f"{title}\n\n{body}",
    }


documents = [parse_document(path) for path in sorted(DATA_DIR.glob("*.txt"))]
docs_df = pd.DataFrame(documents)
docs_df[["doc_id", "title"]]

In [ ]:
# Inspect short previews so it is obvious that the corpus is intentionally messy.
preview_df = docs_df.copy()
preview_df["preview"] = preview_df["body"].str.slice(0, 350) + "..."
preview_df[["title", "preview"]]

## 5. Build the naive vector baseline

This baseline is intentionally weak but realistic:

- one embedding per long document
- no chunking
- no metadata filters
- `top_k=1` retrieval for answer generation and benchmark support

That design tends to collapse when one answer depends on facts scattered across multiple documents.

In [ ]:
vector_index = []

for document in documents:
    vector_index.append(
        {
            "doc_id": document["doc_id"],
            "title": document["title"],
            "text": document["full_text"],
            "embedding": ollama_embed(document["full_text"]),
        }
    )

print(f"Built embeddings for {len(vector_index)} long documents.")


def vector_retrieve(query: str, top_k: int = 1) -> list[dict]:
    """Retrieve semantically similar full documents using a naive document-level strategy."""
    query_embedding = ollama_embed(query)
    scored = []

    for item in vector_index:
        scored.append(
            {
                "doc_id": item["doc_id"],
                "title": item["title"],
                "text": item["text"],
                "score": cosine_similarity(query_embedding, item["embedding"]),
            }
        )

    scored.sort(key=lambda row: row["score"], reverse=True)
    return scored[:top_k]


vector_retrieve(
    "Which supplier is tied to Atlas Gateway shipments, and who receives Atlas Gateway firmware escalations?"
)

## 6. Extract a graph from prose

Real graph pipelines usually rely on an information extraction layer. That layer might be:

- hand-written rules
- an LLM extraction prompt
- a classical NLP pipeline
- a hybrid validation system

To keep the tutorial transparent, we use **deterministic extraction rules** over the prose. The source documents still look realistic, but the graph-building step is explicit and inspectable.

In [ ]:
def add_if_match(triples: list[tuple[str, str, str]], text: str, pattern: str, edges: list[tuple[str, str, str]]):
    """Append fixed triples when a prose pattern is found in a document."""
    if re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL):
        triples.extend(edges)


def extract_triples(document: dict) -> list[tuple[str, str, str]]:
    """Extract a small relationship graph from noisy prose using readable rules."""
    text = document["full_text"]
    triples: list[tuple[str, str, str]] = []

    add_if_match(
        triples,
        text,
        r"Orion Sensor.*captures vibration and temperature telemetry",
        [
            ("Orion Sensor", "CAPTURES", "Vibration Telemetry"),
            ("Orion Sensor", "CAPTURES", "Temperature Telemetry"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"telemetry moves through the Atlas Gateway before it appears in Nimbus Analytics|telemetry is reaching Nimbus Analytics through Atlas Gateway",
        [
            ("Orion Sensor", "SENDS_DATA_TO", "Atlas Gateway"),
            ("Atlas Gateway", "FORWARDS_TO", "Nimbus Analytics"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"Platform Reliability owns Atlas Gateway firmware stability and the operational health of Nimbus Analytics",
        [
            ("Platform Reliability", "OWNS", "Atlas Gateway"),
            ("Platform Reliability", "OWNS", "Nimbus Analytics"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"BlueRiver Circuits provides the controller boards used inside Orion Sensor",
        [("BlueRiver Circuits", "SUPPLIES", "Orion Sensor")],
    )

    add_if_match(
        triples,
        text,
        r"Delta Batteries supplies the backup battery packs required for Atlas Gateway shipments",
        [("Delta Batteries", "SUPPLIES", "Atlas Gateway")],
    )

    add_if_match(
        triples,
        text,
        r"Harbor Logistics moves finished hardware through Chennai Port",
        [("Harbor Logistics", "ROUTES_THROUGH", "Chennai Port")],
    )

    add_if_match(
        triples,
        text,
        r"Bengaluru Warehouse becomes the preferred fallback distribution site",
        [("Bengaluru Warehouse", "BACKS_UP", "Chennai Port")],
    )

    add_if_match(
        triples,
        text,
        r"Two weeks of Orion Sensor safety stock are typically held in Bengaluru",
        [("Bengaluru Warehouse", "STOCKS", "Orion Sensor")],
    )

    add_if_match(
        triples,
        text,
        r"Chennai Port closed for three days",
        [("Monsoon Flooding", "DISRUPTS", "Chennai Port")],
    )

    add_if_match(
        triples,
        text,
        r"Harbor Logistics suspended normal route planning",
        [("Monsoon Flooding", "DISRUPTS", "Harbor Logistics")],
    )

    add_if_match(
        triples,
        text,
        r"Delta Batteries deliveries were delayed",
        [
            ("Monsoon Flooding", "DELAYS", "Delta Batteries"),
            ("Delta Batteries", "IMPACTS", "Atlas Gateway"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"rerouting through the Bengaluru Warehouse to protect Pune customer deployments",
        [
            ("Lumina Manufacturing", "REROUTES_THROUGH", "Bengaluru Warehouse"),
            ("Bengaluru Warehouse", "PROTECTS", "Pune"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"Pune is scheduled before Singapore",
        [
            ("Q3 Launch", "PRIORITIZES", "Pune"),
            ("Q3 Launch", "PRIORITIZES", "Singapore"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"BlueRiver Circuits must maintain controller board output for Orion Sensor",
        [
            ("Q3 Launch", "DEPENDS_ON", "BlueRiver Circuits"),
            ("BlueRiver Circuits", "SUPPORTS", "Orion Sensor"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"Delta Batteries must return to normal shipment volume for Atlas Gateway",
        [
            ("Q3 Launch", "DEPENDS_ON", "Delta Batteries"),
            ("Delta Batteries", "SUPPORTS", "Atlas Gateway"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"dashboard elements are gateway health, sensor drift, and supplier delay signals",
        [
            ("Nimbus Analytics", "DISPLAYS", "Gateway Health"),
            ("Nimbus Analytics", "DISPLAYS", "Sensor Drift"),
            ("Nimbus Analytics", "DISPLAYS", "Supplier Delay Signals"),
        ],
    )

    add_if_match(
        triples,
        text,
        r"Pune customers require a same-day mitigation plan",
        [("Pune Customers", "REQUIRE", "Same Day Mitigation")],
    )

    add_if_match(
        triples,
        text,
        r"Singapore customers usually accept next-business-day mitigation",
        [("Singapore Customers", "ACCEPT", "Next Business Day Mitigation")],
    )

    add_if_match(
        triples,
        text,
        r"Atlas Gateway firmware appears unstable, the incident must be escalated to Platform Reliability",
        [("Atlas Gateway", "ESCALATES_TO", "Platform Reliability")],
    )

    add_if_match(
        triples,
        text,
        r"supplier delays are correlated with rollout risk inside Nimbus Analytics",
        [("Supplier Delays", "OBSERVED_IN", "Nimbus Analytics")],
    )

    return list(dict.fromkeys(triples))


graph = nx.MultiDiGraph()
relation_rows = []
node_to_docs = defaultdict(set)

for document in documents:
    triples = extract_triples(document)
    for source, relation, target in triples:
        graph.add_node(source)
        graph.add_node(target)
        graph.add_edge(source, target, relation=relation, doc_id=document["doc_id"], title=document["title"])
        node_to_docs[source].add(document["doc_id"])
        node_to_docs[target].add(document["doc_id"])
        relation_rows.append(
            {
                "source": source,
                "relation": relation,
                "target": target,
                "doc_id": document["doc_id"],
                "doc_title": document["title"],
            }
        )

relations_df = pd.DataFrame(relation_rows)
print(f"Nodes: {graph.number_of_nodes()} | Edges: {graph.number_of_edges()}")
relations_df

## 7. Graph retrieval over extracted relationships

We use seed-entity matching plus graph traversal. Because the graph is small, we keep the retrieval logic simple and readable.

In production you would likely add:

- entity normalization
- query decomposition
- graph scoring and path ranking
- hybrid retrieval fallback

In [ ]:
def normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value.lower()).strip()


def find_seed_nodes(query: str) -> list[str]:
    """Find graph seed nodes using exact phrase and token-subset matching."""
    query_norm = normalize_text(query)
    query_tokens = set(re.findall(r"[a-zA-Z]+", query_norm))
    seeds = []

    for node in graph.nodes:
        node_norm = normalize_text(node)
        node_tokens = {token for token in re.findall(r"[a-zA-Z]+", node_norm) if len(token) > 2}

        if node_norm in query_norm:
            seeds.append(node)
            continue

        if node_tokens and node_tokens.issubset(query_tokens):
            seeds.append(node)

    return sorted(set(seeds))


def graph_retrieve(query: str, max_hops: int = 2, max_edges: int = 12) -> dict:
    """Traverse the extracted graph to collect connected facts across documents."""
    seed_nodes = find_seed_nodes(query)
    visited_nodes = set(seed_nodes)
    frontier = [(seed, 0) for seed in seed_nodes]
    collected_edges = []

    while frontier and len(collected_edges) < max_edges:
        current_node, depth = frontier.pop(0)
        if depth >= max_hops:
            continue

        for neighbor, edge_bundle in graph[current_node].items():
            for _, edge_data in edge_bundle.items():
                collected_edges.append(
                    {
                        "source": current_node,
                        "relation": edge_data["relation"],
                        "target": neighbor,
                        "doc_id": edge_data["doc_id"],
                        "doc_title": edge_data["title"],
                    }
                )
                if neighbor not in visited_nodes:
                    visited_nodes.add(neighbor)
                    frontier.append((neighbor, depth + 1))

        for predecessor in graph.predecessors(current_node):
            for _, edge_data in graph[predecessor][current_node].items():
                collected_edges.append(
                    {
                        "source": predecessor,
                        "relation": edge_data["relation"],
                        "target": current_node,
                        "doc_id": edge_data["doc_id"],
                        "doc_title": edge_data["title"],
                    }
                )
                if predecessor not in visited_nodes:
                    visited_nodes.add(predecessor)
                    frontier.append((predecessor, depth + 1))

    unique_edges = []
    seen = set()
    for edge in collected_edges:
        edge_key = (edge["source"], edge["relation"], edge["target"], edge["doc_id"])
        if edge_key not in seen:
            unique_edges.append(edge)
            seen.add(edge_key)

    supporting_doc_ids = []
    for edge in unique_edges:
        if edge["doc_id"] not in supporting_doc_ids:
            supporting_doc_ids.append(edge["doc_id"])

    doc_lookup = {doc["doc_id"]: doc for doc in documents}
    doc_snippets = []
    for doc_id in supporting_doc_ids:
        snippet = doc_lookup[doc_id]["body"][:350].replace("\n", " ")
        doc_snippets.append(f"[{doc_id}] {snippet}...")

    edge_lines = [
        f"{edge['source']} -> {edge['relation']} -> {edge['target']} (source: {edge['doc_id']})"
        for edge in unique_edges[:max_edges]
    ]

    return {
        "query": query,
        "seed_nodes": seed_nodes,
        "edges": unique_edges[:max_edges],
        "doc_ids": supporting_doc_ids,
        "context": "Structured facts:\n"
        + "\n".join(edge_lines)
        + "\n\nSupporting document snippets:\n"
        + "\n".join(doc_snippets),
    }


graph_retrieve(
    "Which supplier is tied to Atlas Gateway shipments, and who receives Atlas Gateway firmware escalations?"
)

## 8. Benchmark questions designed to stress document-level vector search

These questions are intentionally multi-hop. Each answer depends on relationships that are scattered across the corpus.

That makes them a poor fit for a one-document vector baseline and a much better fit for a graph traversal layer.

In [ ]:
benchmark_questions = [
    {
        "query": "Which supplier is tied to Atlas Gateway shipments, and who receives Atlas Gateway firmware escalations?",
        "expected_terms": ["delta batteries", "platform reliability"],
    },
    {
        "query": "Which supplier underpins Orion Sensor, and which market is launched before Singapore?",
        "expected_terms": ["blueriver circuits", "pune"],
    },
    {
        "query": "Which application should launch managers use for gateway health and supplier delay signals, and who owns its operational health?",
        "expected_terms": ["nimbus analytics", "platform reliability"],
    },
    {
        "query": "When Chennai routing is impaired, which fallback site protects Pune commitments and which logistics partner was disrupted?",
        "expected_terms": ["bengaluru warehouse", "harbor logistics"],
    },
    {
        "query": "Which component sits between Orion Sensor and Nimbus Analytics, and which team owns that component's firmware stability?",
        "expected_terms": ["atlas gateway", "platform reliability"],
    },
    {
        "query": "Which customer group requires same-day mitigation, and which supplier delay can impact Atlas Gateway?",
        "expected_terms": ["pune customers", "delta batteries"],
    },
]


def supports_expected_terms(text: str, expected_terms: list[str]) -> bool:
    normalized_text = normalize_text(text)
    return all(term.lower() in normalized_text for term in expected_terms)


results = []

for item in benchmark_questions:
    query = item["query"]

    vector_start = time.perf_counter()
    vector_hits = vector_retrieve(query, top_k=1)
    vector_latency_ms = (time.perf_counter() - vector_start) * 1000
    vector_context = "\n\n".join(hit["text"] for hit in vector_hits)

    graph_start = time.perf_counter()
    graph_hits = graph_retrieve(query, max_hops=2, max_edges=12)
    graph_latency_ms = (time.perf_counter() - graph_start) * 1000
    graph_context = graph_hits["context"]

    results.append(
        {
            "query": query,
            "vector_docs": [hit["doc_id"] for hit in vector_hits],
            "graph_docs": graph_hits["doc_ids"],
            "graph_seed_nodes": graph_hits["seed_nodes"],
            "vector_latency_ms": round(vector_latency_ms, 2),
            "graph_latency_ms": round(graph_latency_ms, 2),
            "vector_supports_answer": supports_expected_terms(vector_context, item["expected_terms"]),
            "graph_supports_answer": supports_expected_terms(graph_context, item["expected_terms"]),
        }
    )

results_df = pd.DataFrame(results)
results_df

In [ ]:
summary_df = pd.DataFrame(
    [
        {
            "method": "vector_rag_top1_full_docs",
            "avg_latency_ms": results_df["vector_latency_ms"].mean(),
            "support_rate": results_df["vector_supports_answer"].mean(),
        },
        {
            "method": "graph_rag_extracted_relations",
            "avg_latency_ms": results_df["graph_latency_ms"].mean(),
            "support_rate": results_df["graph_supports_answer"].mean(),
        },
    ]
).round(3)

summary_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

latency_df = summary_df.set_index("method")[["avg_latency_ms"]]
latency_df.plot(kind="bar", ax=axes[0], legend=False)
axes[0].set_title("Average retrieval latency")
axes[0].set_xlabel("")
axes[0].set_ylabel("Latency (ms)")

support_df = summary_df.set_index("method")[["support_rate"]]
support_df.plot(kind="bar", ax=axes[1], legend=False)
axes[1].set_title("Answer-support rate")
axes[1].set_xlabel("")
axes[1].set_ylabel("Rate")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 9. Generate answers with both methods

The benchmark above checks whether the retrieved context contains the expected facts. Now we pass that context to the local generation model.

In [ ]:
def build_answer_prompt(method_name: str, query: str, context: str) -> str:
    """Create the same grounding prompt for both retrieval methods."""
    return textwrap.dedent(
        f"""
        You are answering internal Lumina Manufacturing questions.
        Use only the supplied context.
        If the context is incomplete, say so directly.

        Retrieval method: {method_name}
        User question: {query}

        Context:
        {context}

        Answer in 3-5 sentences.
        """
    ).strip()


def answer_with_vector_rag(query: str) -> dict:
    hits = vector_retrieve(query, top_k=1)
    context = "\n\n".join(hit["text"] for hit in hits)
    prompt = build_answer_prompt("vector_rag_top1_full_docs", query, context)
    return {
        "docs": [hit["doc_id"] for hit in hits],
        "context": context,
        "answer": ollama_generate(prompt),
    }


def answer_with_graph_rag(query: str) -> dict:
    retrieval = graph_retrieve(query, max_hops=2, max_edges=12)
    prompt = build_answer_prompt("graph_rag_extracted_relations", query, retrieval["context"])
    return {
        "docs": retrieval["doc_ids"],
        "seed_nodes": retrieval["seed_nodes"],
        "context": retrieval["context"],
        "answer": ollama_generate(prompt),
    }


demo_query = "Which supplier is tied to Atlas Gateway shipments, and who receives Atlas Gateway firmware escalations?"

vector_demo = answer_with_vector_rag(demo_query)
graph_demo = answer_with_graph_rag(demo_query)

pd.DataFrame(
    [
        {
            "method": "vector_rag_top1_full_docs",
            "docs": vector_demo["docs"],
            "answer": vector_demo["answer"],
        },
        {
            "method": "graph_rag_extracted_relations",
            "docs": graph_demo["docs"],
            "answer": graph_demo["answer"],
        },
    ]
)

## 10. Interpreting the result fairly

If the notebook behaves as intended, the vector baseline should look poor on this benchmark. That is not because vector search is useless. It is because the baseline is underpowered for the problem.

What this tutorial actually demonstrates is:

- full-document vector retrieval is weak for dependency-heavy enterprise questions
- graph retrieval can recover scattered facts once relationships are explicit
- relationship-centric workloads are a strong reason to add graph RAG or a hybrid retriever

If you upgraded the vector side with chunking, metadata filters, better query decomposition, and multi-document fusion, the comparison would become more competitive. That is exactly the point: **graph RAG is the next step when naive vector retrieval runs out of road**.

## 11. How this maps to production

A real deployment should separate ingestion, extraction, storage, and serving.

```mermaid
flowchart TB
    A[Policies, Playbooks, Notes, Incident Docs] --> B[Ingestion Pipeline]
    B --> C[Chunking + Metadata]
    B --> D[Entity and Relationship Extraction]
    C --> E[Vector Database]
    D --> F[Graph Database]
    G[Query API] --> H[Retriever Router]
    H --> E
    H --> F
    E --> I[Context Builder]
    F --> I
    I --> J[LLM Gateway]
    J --> K[Answer + Observability Trace]
```

### Production upgrades

1. Replace the notebook vector baseline with chunked retrieval in `pgvector`, Qdrant, Weaviate, or Milvus.
2. Replace the in-memory graph with Neo4j, Neptune, or another graph store.
3. Move extraction into a repeatable ingestion pipeline and validate extracted edges before serving them.
4. Add source attribution, latency monitoring, evaluation harnesses, and access control.
5. Keep the Ollama path for local development, then swap to a production model gateway behind the same interfaces.

A practical architecture often ends up hybrid:

- semantic questions route to vector retrieval
- dependency and ownership questions route to graph retrieval
- high-value workflows use both

## 12. Takeaway

For messy company documents, naive vector retrieval is often the first step and often the wrong stopping point. Once questions depend on supplier links, escalation ownership, routing fallback, and cross-document dependencies, graph RAG stops being an academic extra and starts looking like the correct next layer.